# Assignment 4 — Urban rainfall monitoring

You have been hired by the city technical office to monitor intense urban rainfall in Lugano, close to the Cassarate catchment and a dense urban drainage area. The objective is to support early warning for pluvial flooding and sewer overload. The node is installed on a public building roof and simulates a rain gauge combined with sensors describing accumulated rainfall and drainage saturation.

**Location:** Lugano city centre / Cassarate urban catchment, Canton Ticino, Switzerland  
**Thing to register in istSOS4:** `Urban rainfall station LUG-RAIN-01`  
**Sensor:** Tipping-bucket rain gauge with drainage level sensor


## Assignment

Design and register a SensorThings/istSOS4 monitoring setup for this case. Then run the real-time generator and send the simulated observations to the correct Datastreams.

Each generated row has this form:

```python
[phenomenonTime, value_1, value_2, value_3]
```

The generator does **not** assign quality flags. Each value starts as raw data (`0`) and your code must classify it using the thresholds below.

| Quality | Meaning | Rule |
|---:|---|---|
| 0 | raw | value not yet checked |
| 1 | sensible | value inside statistical thresholds |
| 2 | suspect | value outside statistical thresholds but still inside plausibility thresholds |
| 3 | alarm | value outside plausibility thresholds |
| -999 | invalid | value outside physical limits |

| Parameter | Meaning | Unit | Physical limit | Plausibility threshold | Statistical threshold |
|---|---|---:|---:|---:|---:|
| `rainfall_intensity_mm_h` | Short-term rainfall intensity | mm/h | 0 – 300 | 0 – 180 | 0 – 35 |
| `cumulated_rainfall_mm` | Cumulated rainfall since the beginning of the event | mm | 0 – 400 | 0 – 250 | 0 – 45 |
| `drain_level_pct` | Urban drainage filling level | % | 0 – 150 | 0 – 120 | 0 – 70 |



## Generator behaviour

The generator runs in real time when `sleep=True`. With `step_seconds=30`, one row is produced every 30 seconds.

Recommended settings for the assignment:

```python
alarm_after_seconds=600       # first alarm after about 10 minutes
alarm_duration_seconds=180    # alarm lasts about 3 minutes
alarm_repeat_seconds=None     # no repetition
```

If you want repeated events during a longer exercise, set for example:

```python
alarm_repeat_seconds=900
```

This means that, after the first alarm start, a new alarm window starts every 900 seconds. If it is left as `None`, the alarm happens only once.


In [ ]:
QUALITY_RAW = 0
QUALITY_SENSIBLE = 1
QUALITY_SUSPECT = 2
QUALITY_ALARM = 3


In [ ]:
from scripts.sensor_stream_generators import UrbanRainfallGenerator

generator = UrbanRainfallGenerator(
    step_seconds=30,
    alarm_after_seconds=600,
    alarm_duration_seconds=180,
    alarm_repeat_seconds=None,  # set e.g. 900 to repeat alarms every 15 minutes
    suspect_probability=0.08,
    seed=7
)

# login to istSOS and create entities here if needed


# logout and login as ddt_sensor (pwd: qwertz) to push data with the sensor token
print(generator.cols())
for i in range(10):  # adjust the range as needed
    row = generator.read(sleep=False)  # use sleep=True in the real acquisition loop
    print(row)
    print("simulated status:", generator.status())
    # quality flags

    # push to istSOS or other system here


